# Week 7: From Baselines to Targeted Experiments

## Objective
In this notebook, we train tree-based models on the cleaned UHPC dataset using a train/validation/test split, then run targeted experiments.

## Goals
- Separate features and target
- Build train/validation/test split
- Train Random Forest and Extra Trees
- Compare results using R2 and RMSE
- Run targeted experiments and justify the baseline

# Step 1: Import Libraries

In this step, we import all the libraries needed for loading data, splitting, training models, and scoring.

In [1]:
# Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.tree import DecisionTreeRegressor

print("Libraries loaded successfully")

Libraries loaded successfully


# Step 2: Load the Cleaned Dataset

In this step, we load the cleaned UHPC dataset created in Week 5. This data is already cleaned, so we do not clean it again.

In [2]:
# Load the cleaned UHPC dataset from Week 5

df = pd.read_excel("week 6/UHPC_Cleaned_Dataset.xlsx")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (2187, 41)


,Unnamed: 0,Cement Amount (kg/m³),Cement type\t,Cement Grade (Mpa),Silica Fume (kg/m³),Flayash Amount (kg/m³),Limestone Powder (kg/m3),Quartzpowder (kg/m3),Glass powder (kg/m3),Rice husk ash (kg/m3),...,Unnamed: 31,Unnamed: 32,Unnamed: 33,Specimen,Testing Standard,7-Day,28-Day,Specimens,Testing Standard .1,Peak Strength (MPa)
0,S10-F5-N5,839.0,Type I/II low-alkali portland cement,NaN,104.0,52.0,0.0,0.0,0.0,0.0,...,NaN,ASTM C143 for Slump\nASTM C1611 for Slump Flow,370.0,100 x 100 x 100,BS1881,104.5,132.0,76×102×406,ASTM C1609,21.25
1,S10-F2.5-N7.5,839.0,Type I/II low-alkali portland cement,NaN,104.0,26.0,0.0,0.0,0.0,0.0,...,NaN,ASTM C143 for Slump\nASTM C1611 for Slump Flow,370.0,100 x 100 x 100,BS1881,102.5,122.5,76×102×406,ASTM C1609,20.80
2,S10-N10,839.0,Type I/II low-alkali portland cement,NaN,104.0,0.0,0.0,0.0,0.0,0.0,...,NaN,ASTM C143 for Slump\nASTM C1611 for Slump Flow,370.0,100 x 100 x 100,BS1881,100.5,116.0,76×102×406,ASTM C1609,20.00
3,S10-F5-M5,839.0,Type I/II low-alkali portland cement,NaN,104.0,52.0,0.0,0.0,0.0,0.0,...,NaN,ASTM C143 for Slump\nASTM C1611 for Slump Flow,380.0,100 x 100 x 100,BS1881,108.5,134.0,76×102×406,ASTM C1609,20.90
4,S10-F2.5-M7.5,839.0,Type I/II low-alkali portland cement,NaN,104.0,26.0,0.0,0.0,0.0,0.0,...,NaN,ASTM C143 for Slump\nASTM C1611 for Slump Flow,370.0,100 x 100 x 100,BS1881,108.5,131.5,76×102×406,ASTM C1609,20.50


# Step 3: Separate Features and Target

In this step, we split the data into features (X) and target (y).

- Target (y) = 28-Day compressive strength
- Features (X) = mix ingredients
- We remove junk columns: row numbers, empty columns, and other results

We exclude arbitrary identifiers and other strength outcomes, keeping only the mix ingredients as predictors.

In [3]:
# Junk columns we remove (not useful for prediction)
drop_columns = [
    "Unnamed: 0",
    "Unnamed: 29", "Unnamed: 30", "Unnamed: 31",
    "Unnamed: 32", "Unnamed: 33",
    "Specimen ", "Testing Standard ",
    "7-Day", "Specimens ", "Testing Standard .1",
    "Peak Strength (MPa)"
]

# Target column we want to predict
target_column = "28-Day"

# Features (X) and target (y)
X = df.drop(columns=drop_columns + [target_column])
y = df[target_column]

# Drop rows where target is missing (professor instruction)
missing_target = y.isnull()
X = X[~missing_target]
y = y[~missing_target]

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Missing in y:", y.isnull().sum())

Features shape: (2072, 28)
Target shape: (2072,)
Missing in y: 0


In [4]:
# Check which columns are text (object type)

print(X.dtypes[X.dtypes == "object"])

Max Size (mm)    object
dtype: object


In [5]:
# See the unique values in Max Size column

print(X["Max Size (mm)"].unique())

[nan 0.8 '0.30\u2009' 0.6 0.3 0.45 0.1 3 4.75 2.36 2.5 1.2 0.337 2 1.25
 0.27 1 0.85 1.18 4 0.075 0.35 0.275 0.225 0.63 0.181 0.18 0.2 0.22 0.9
 0.5 0.15 2.62 0.045 0.1556 1.12 1.7]


# Step 4: Check Missing Values

In this step, we check how many missing values are in the features. Models cannot train on missing values, so we must know how many there are before handling them.

In [6]:
# Check missing values in features

missing = X.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("Columns with missing values:")
print(missing)

Columns with missing values:
Tensile Strength (MPa)         1182
Cement Grade (Mpa)              798
Length (mm)                     676
Diameter (mm)                   675
Type of Fiber                   669
Amount / Quantity of Fiber      669
Max Size (mm)                   276
Type of Superplasticizer        120
Sand Type                        81
Cement type\t                    19
Amount  (kg/m³).2                 8
dtype: int64


# Step 5: Handle Missing Values

In this step, we fill missing values using semantic logic.

- Number columns: blank means the ingredient was not added, so we fill with 0
- Text columns: blank means no such material was used, so we fill with "None"

In [7]:
# Separate columns into number type and text type

number_cols = X.select_dtypes(include="number").columns
text_cols = X.select_dtypes(include=["object", "string"]).columns

# Fill number columns with 0
X[number_cols] = X[number_cols].fillna(0)

# Fill text columns with "None"
X[text_cols] = X[text_cols].fillna("None")

# Check if any missing left
print("Missing values left:", X.isnull().sum().sum())

Missing values left: 0


# Step 6: Check Text Column Labels

- Count unique labels in each text column
- Columns with too many labels are messy and will be dropped
- Columns with few labels will be encoded

In [8]:
# Check how many unique labels each text column has

for col in text_cols:
    print(col, "→", X[col].nunique(), "labels")

Cement type	 → 144 labels
Sand Type → 34 labels
Max Size (mm) → 37 labels
Type of Fiber  → 53 labels
Type of Superplasticizer → 128 labels


# Step 7: Drop High-Cardinality Text Columns

- These text columns have too many unique labels (34 to 145)
- Encoding them creates sparse, high-dimensional, noisy data
- Trade-off: we drop them to keep the model clean and reliable
- We keep all numeric ingredient features

In [9]:
# Drop the high-cardinality text columns

X = X.drop(columns=text_cols)

print("Shape after dropping text columns:", X.shape)
X.head()

Shape after dropping text columns: (2072, 23)


,Cement Amount (kg/m³),Cement Grade (Mpa),Silica Fume (kg/m³),Flayash Amount (kg/m³),Limestone Powder (kg/m3),Quartzpowder (kg/m3),Glass powder (kg/m3),Rice husk ash (kg/m3),Metakaolin (kg/m³),GGBFS (kg/m³),...,Nano-TiO2 (kg/m3),Nano Silica (kg/m3),Filler (kg/m³),Amount (kg/m³),Amount / Quantity of Fiber,Length (mm),Diameter (mm),Tensile Strength (MPa),Amount (kg/m³).1,Amount (kg/m³).2
0,839.0,0.0,104.0,52.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,52.0,925.0,119.0,13.0,0.2,1900.0,147.0,59.33
1,839.0,0.0,104.0,26.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,78.0,931.0,119.0,13.0,0.2,1900.0,147.0,59.33
2,839.0,0.0,104.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,104.0,931.0,119.0,13.0,0.2,1900.0,147.0,62.15
3,839.0,0.0,104.0,52.0,0.0,0.0,0.0,0.0,52.0,0.0,...,0.0,0.0,0.0,913.0,119.0,13.0,0.2,1900.0,147.0,64.98
4,839.0,0.0,104.0,26.0,0.0,0.0,0.0,0.0,78.0,0.0,...,0.0,0.0,0.0,921.0,119.0,13.0,0.2,1900.0,147.0,64.98


# Step 8: Train / Validation / Test Split

- Split data into 3 parts: train (60%), validation (20%), test (20%)
- Train: model learns from this
- Validation: used to tune and compare models
- Test: final honest score, used only once

In [10]:
# Step 1: split off the test set (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Step 2: split remaining 80% into train (60%) and validation (20%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42
)

print("Train size:     ", X_train.shape[0])
print("Validation size:", X_val.shape[0])
print("Test size:      ", X_test.shape[0])

Train size:      1242
Validation size: 415
Test size:       415


# Step 9: Decision Tree (Baseline)

- Decision Tree is a single tree, our simplest model
- It is the baseline to compare other models against
- We train on training data and check score on validation data

In [11]:
# Train the Decision Tree (our baseline model)

dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)

# Predict on validation data
dt_val_pred = dt_model.predict(X_val)

# Check the scores
dt_r2 = r2_score(y_val, dt_val_pred)
dt_rmse = np.sqrt(mean_squared_error(y_val, dt_val_pred))

print("Decision Tree (Baseline)")
print("Validation R2:  ", round(dt_r2, 4))
print("Validation RMSE:", round(dt_rmse, 4))

Decision Tree (Baseline)
Validation R2:   0.5002
Validation RMSE: 26.6834


# Step 10: Random Forest Model

- Random Forest builds 100 decision trees and averages their predictions
- It reduces overfitting compared to a single Decision Tree
- We compare its performance against the Decision Tree baseline

In [14]:
# Train Random Forest model

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict on validation data
rf_val_pred = rf_model.predict(X_val)

# Check the scores
rf_r2 = r2_score(y_val, rf_val_pred)
rf_rmse = np.sqrt(mean_squared_error(y_val, rf_val_pred))

print("Random Forest")
print("Validation R2:  ", round(rf_r2, 4))
print("Validation RMSE:", round(rf_rmse, 4))

Random Forest
Validation R2:   0.6601
Validation RMSE: 22.0028


# Step 11: Extra Trees Model

- Extra Trees is similar to Random Forest but uses random split thresholds
- Extra randomness reduces overfitting further and trains faster
- We compare its performance against Decision Tree and Random Forest

In [15]:
# Train Extra Trees model

et_model = ExtraTreesRegressor(n_estimators=100, random_state=42)
et_model.fit(X_train, y_train)

# Predict on validation data
et_val_pred = et_model.predict(X_val)

# Check the scores
et_r2 = r2_score(y_val, et_val_pred)
et_rmse = np.sqrt(mean_squared_error(y_val, et_val_pred))

print("Extra Trees")
print("Validation R2:  ", round(et_r2, 4))
print("Validation RMSE:", round(et_rmse, 4))

Extra Trees
Validation R2:   0.6103
Validation RMSE: 23.5595


# Step 12: Model Comparison

- We compare all three models on validation data
- Decision Tree is the baseline
- Random Forest and Extra Trees are ensemble improvements
- Best model will be evaluated on the test set

In [16]:
# Compare all three models

results = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest", "Extra Trees"],
    "Validation R2": [dt_r2, rf_r2, et_r2],
    "Validation RMSE": [dt_rmse, rf_rmse, et_rmse]
})

print(results.to_string(index=False))

        Model  Validation R2  Validation RMSE
Decision Tree       0.500152        26.683374
Random Forest       0.660129        22.002844
  Extra Trees       0.610337        23.559524


# Step 13: Final Evaluation on Test Set

- Random Forest is our best model (R2: 0.66)
- We now evaluate it on the test set
- Test set was never seen by the model before
- This gives us the final honest score

In [17]:
# Evaluate best model (Random Forest) on test set

rf_test_pred = rf_model.predict(X_test)

rf_test_r2 = r2_score(y_test, rf_test_pred)
rf_test_rmse = np.sqrt(mean_squared_error(y_test, rf_test_pred))

print("Final Test Results — Random Forest")
print("Test R2:  ", round(rf_test_r2, 4))
print("Test RMSE:", round(rf_test_rmse, 4))

Final Test Results — Random Forest
Test R2:   0.7036
Test RMSE: 20.0371


# Step 14: Targeted Experiment — Fiber vs No Fiber

- We investigate if the model performs differently on mixes with and without fiber
- Fiber affects concrete strength significantly
- We split test set into fiber and no-fiber groups and compare RMSE

In [20]:
# Targeted Experiment: Fiber vs No Fiber

# Create test dataframe with predictions
test_df = X_test.copy()
test_df["actual"] = y_test.values
test_df["predicted"] = rf_test_pred

# Split into fiber and no-fiber groups using correct column name
fiber_mask = test_df["Amount / Quantity of Fiber "] > 0

fiber_group = test_df[fiber_mask]
no_fiber_group = test_df[~fiber_mask]

# Calculate RMSE for each group
fiber_rmse = np.sqrt(mean_squared_error(
    fiber_group["actual"], fiber_group["predicted"]
))
no_fiber_rmse = np.sqrt(mean_squared_error(
    no_fiber_group["actual"], no_fiber_group["predicted"]
))

print("Fiber mixes    — RMSE:", round(fiber_rmse, 4))
print("No Fiber mixes — RMSE:", round(no_fiber_rmse, 4))
print()
print("Fiber mixes count:   ", len(fiber_group))
print("No Fiber mixes count:", len(no_fiber_group))

Fiber mixes    — RMSE: 20.8257
No Fiber mixes — RMSE: 18.3898

Fiber mixes count:    275
No Fiber mixes count: 140


# Step 15: Targeted Experiment — High vs Low Strength

- We investigate if the model performs differently on high and low strength mixes
- High strength: above 150 MPa
- Low strength: below 150 MPa
- We compare RMSE for both groups

In [21]:
# Targeted Experiment: High vs Low Strength

# Split test set into high and low strength groups
high_mask = test_df["actual"] >= 150

high_group = test_df[high_mask]
low_group = test_df[~high_mask]

# Calculate RMSE for each group
high_rmse = np.sqrt(mean_squared_error(
    high_group["actual"], high_group["predicted"]
))
low_rmse = np.sqrt(mean_squared_error(
    low_group["actual"], low_group["predicted"]
))

print("High Strength (>=150 MPa) — RMSE:", round(high_rmse, 4))
print("Low Strength  (<150 MPa)  — RMSE:", round(low_rmse, 4))
print()
print("High strength count:", len(high_group))
print("Low strength count: ", len(low_group))

High Strength (>=150 MPa) — RMSE: 23.3505
Low Strength  (<150 MPa)  — RMSE: 16.7004

High strength count: 191
Low strength count:  224


# Step 16: Final Summary

- Decision Tree baseline: R2 = 0.50, RMSE = 26.68 MPa
- Random Forest: R2 = 0.66 (validation), 0.70 (test), RMSE = 20.04 MPa
- Extra Trees: R2 = 0.61, RMSE = 23.56 MPa
- Random Forest is the best model
- Model struggles with fiber mixes and high strength mixes
- Future work: better fiber encoding, more high strength data

In [23]:
# Final Summary of all results

print("WEEK 7 — FINAL RESULTS SUMMARY")
print()
print("Model Comparison on Validation Data:")
print(f"  Decision Tree  R2: {dt_r2:.4f}   RMSE: {dt_rmse:.2f} MPa")
print(f"  Random Forest  R2: {rf_r2:.4f}   RMSE: {rf_rmse:.2f} MPa")
print(f"  Extra Trees    R2: {et_r2:.4f}   RMSE: {et_rmse:.2f} MPa")
print()
print("Best Model — Random Forest on Test Set:")
print(f"  Test R2:   {rf_test_r2:.4f}")
print(f"  Test RMSE: {rf_test_rmse:.2f} MPa")
print()
print("Targeted Experiments:")
print(f"  Fiber mixes     RMSE: {fiber_rmse:.2f} MPa")
print(f"  No Fiber mixes  RMSE: {no_fiber_rmse:.2f} MPa")
print(f"  High Strength   RMSE: {high_rmse:.2f} MPa")
print(f"  Low Strength    RMSE: {low_rmse:.2f} MPa")
print()
print("Key Findings:")
print("  1. Random Forest outperforms Decision Tree and Extra Trees")
print("  2. Model predicts no-fiber mixes more accurately")
print("  3. High strength mixes above 150 MPa are harder to predict")
print("  4. Future work: improve fiber type encoding")

WEEK 7 — FINAL RESULTS SUMMARY

Model Comparison on Validation Data:
  Decision Tree  R2: 0.5002   RMSE: 26.68 MPa
  Random Forest  R2: 0.6601   RMSE: 22.00 MPa
  Extra Trees    R2: 0.6103   RMSE: 23.56 MPa

Best Model — Random Forest on Test Set:
  Test R2:   0.7036
  Test RMSE: 20.04 MPa

Targeted Experiments:
  Fiber mixes     RMSE: 20.83 MPa
  No Fiber mixes  RMSE: 18.39 MPa
  High Strength   RMSE: 23.35 MPa
  Low Strength    RMSE: 16.70 MPa

Key Findings:
  1. Random Forest outperforms Decision Tree and Extra Trees
  2. Model predicts no-fiber mixes more accurately
  3. High strength mixes above 150 MPa are harder to predict
  4. Future work: improve fiber type encoding
